# Kaggle Notebook Template: Registry Bundle (Scrap Classifier)

This notebook trains a scrap classifier bundle compatible with backend registry loading in `data_access._resolve_task_bundle(...)`.

It exports one artifact payload (joblib dict) containing:

- `model`
- `scaler`
- `feature_cols`
- `feature_spec`
- `feature_spec_hash`
- `family`
- `task`
- `decision_threshold`
- `metrics`

Then you can import/promote locally with `backend_fastapi/import_registry_bundle.py`.

In [ ]:
!pip -q install lightgbm scikit-learn joblib openpyxl pandas numpy

## Configure Paths

In [ ]:
from pathlib import Path

# Adjust these paths for your Kaggle datasets
REPO_INPUT_DIR = Path('/kaggle/input/predictive-scrap-ai')
DATA_DIR = Path('/kaggle/input/smart-factory-data')
MES_WORKBOOK = DATA_DIR / 'MES_Manufacturing_M-231_M-356_M-471_M-607_M-612.xlsx'
PARAMETER_CSV = REPO_INPUT_DIR / 'backend_fastapi' / 'AI_cup_parameter_info_cleaned.csv'

# Optional: force subset of files; leave empty for auto-discovery M*.csv
MACHINE_FILES = []

# Choose model id you will register locally
MODEL_ID = 'scrap_classifier__lightgbm__kaggle__v1'

OUTPUT_ROOT = Path('/kaggle/working/sfb_registry_export')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE_PATH = OUTPUT_ROOT / f'{MODEL_ID}.pkl'
MANIFEST_PATH = OUTPUT_ROOT / 'registry_bundle_manifest.json'
print('OUTPUT_ROOT =', OUTPUT_ROOT)

In [ ]:
# Load project training helpers directly from repo input
import importlib.util

trainer_path = REPO_INPUT_DIR / 'backend_fastapi' / 'train_models_from_csv.py'
if not trainer_path.exists():
    raise FileNotFoundError(f'Missing trainer module: {trainer_path}')

spec = importlib.util.spec_from_file_location('train_models_from_csv', trainer_path)
trainer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(trainer)

print('Loaded:', trainer_path)

## Build Training Frame

In [ ]:
import numpy as np
import pandas as pd

if not DATA_DIR.exists():
    raise FileNotFoundError(f'Missing DATA_DIR: {DATA_DIR}')
if not PARAMETER_CSV.exists():
    raise FileNotFoundError(f'Missing PARAMETER_CSV: {PARAMETER_CSV}')

parameter_df = trainer.enrich_parameter_csv(PARAMETER_CSV, OUTPUT_ROOT / 'AI_cup_parameter_info_cleaned_v2.csv')

machine_files = MACHINE_FILES if MACHINE_FILES else sorted([p.name for p in DATA_DIR.glob('M*.csv')])
if not machine_files:
    raise RuntimeError('No machine files discovered. Add M*.csv or set MACHINE_FILES.')

frame, feature_cols, data_sources = trainer.build_minute_horizon_training_frame(
    data_dir=DATA_DIR,
    machine_files=machine_files,
    mes_workbook=MES_WORKBOOK,
    parameter_df=parameter_df,
    max_rows_per_machine=300000,
    chunk_size=50000,
)

if frame.empty:
    raise RuntimeError('Training frame is empty.')

print('rows=', len(frame), 'feature_count=', len(feature_cols), 'sources=', data_sources)

## Train Classifier Bundle

Uses `scrap_event_30m` target and a chronological split.

In [ ]:
import hashlib
import json
from datetime import datetime, timezone

import joblib
import lightgbm as lgb
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

label_col = 'scrap_event_30m'
if label_col not in frame.columns:
    raise RuntimeError(f'Missing target column: {label_col}')

subset = frame.dropna(subset=[label_col]).copy()
subset = subset.sort_values('timestamp').reset_index(drop=True)
if len(subset) < 200:
    raise RuntimeError(f'Not enough rows after filtering: {len(subset)}')

X = subset[feature_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0.0)
y = subset[label_col].astype(int)

split_idx = max(1, int(len(subset) * 0.8))
X_train = X.iloc[:split_idx]
X_val = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_val = y.iloc[split_idx:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val) if len(X_val) else X_val.to_numpy(dtype=float)

if len(set(y_train.tolist())) < 2:
    model = DummyClassifier(strategy='most_frequent')
    model.fit(X_train_scaled, y_train)
    family = 'dummy_classifier'
else:
    model = lgb.LGBMClassifier(
        objective='binary',
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=1,
        verbosity=-1,
    )
    model.fit(X_train_scaled, y_train)
    family = 'lightgbm'

if len(X_val) > 0:
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_val_scaled)
        if proba.ndim == 2 and proba.shape[1] > 1:
            val_prob = proba[:, 1]
        else:
            val_prob = np.asarray(proba).reshape(-1)
    else:
        val_prob = np.asarray(model.predict(X_val_scaled)).reshape(-1)
else:
    val_prob = np.array([], dtype=float)

metrics = {
    'rows_train': int(len(X_train)),
    'rows_val': int(len(X_val)),
    'positive_rate_train': float(y_train.mean()) if len(y_train) else 0.0,
    'positive_rate_val': float(y_val.mean()) if len(y_val) else 0.0,
}
if len(X_val) > 0 and len(set(y_val.tolist())) > 1 and len(val_prob) == len(y_val):
    metrics['roc_auc'] = float(roc_auc_score(y_val, val_prob))
    metrics['pr_auc'] = float(average_precision_score(y_val, val_prob))

feature_spec = {
    'source': 'kaggle_registry_bundle_training_template',
    'label_col': label_col,
    'data_sources': data_sources,
}
feature_spec_hash = hashlib.sha256(
    json.dumps({'feature_cols': list(feature_cols), 'feature_spec': feature_spec}, sort_keys=True).encode('utf-8')
).hexdigest()

artifact_payload = {
    'model': model,
    'scaler': scaler,
    'feature_cols': list(feature_cols),
    'feature_spec': feature_spec,
    'feature_spec_hash': feature_spec_hash,
    'family': family,
    'task': 'scrap_classifier',
    'decision_threshold': 0.5,
    'metrics': metrics,
    'trained_at': datetime.now(timezone.utc).isoformat(),
}
joblib.dump(artifact_payload, BUNDLE_PATH)

manifest = {
    'ok': True,
    'model_id': MODEL_ID,
    'task': 'scrap_classifier',
    'family': family,
    'artifact_file': BUNDLE_PATH.name,
    'feature_spec_hash': feature_spec_hash,
    'metrics': metrics,
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))

In [ ]:
# Zip export for download
import shutil

zip_base = '/kaggle/working/sfb_registry_bundle_export'
zip_path = shutil.make_archive(base_name=zip_base, format='zip', root_dir=OUTPUT_ROOT)
print('Created zip:', zip_path)
print('Download from Kaggle outputs panel.')

## Local Import + Promote Commands

After downloading and extracting locally, run from project root:

```powershell
python backend_fastapi/import_registry_bundle.py \
  --bundle-joblib ".\\downloaded\\<MODEL_ID>.pkl" \
  --model-id "<MODEL_ID>" \
  --task scrap_classifier \
  --promote
```

Then verify:

- `GET /api/admin/models`
- `GET /api/health`
- dashboard future risk endpoints